In [0]:
data = [
    ("U001", "2024-01-01"),
    ("U001", "2024-01-02"),
    ("U001", "2024-01-04"),
    ("U001", "2024-01-07"),
    ("U002", "2024-03-15"),
    ("U002", "2024-03-16"),
    ("U002", "2024-03-18"),
]

columns = ["user_id", "login_date"]

df = spark.createDataFrame(data, columns)
display(df)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

df = df.withColumn('login_date', to_date('login_date'))
df.dtypes

In [0]:
#window_spec = Window.partitionBy('user_id').orderBy('recharge_date')
df_dates = df.groupBy("user_id").agg(min('login_date').alias('min_date'),
                               max("login_date").alias('max_date'))\
             .withColumn('all_dates',explode(sequence('min_date','max_date')))\
             .drop('min_date','max_date')

df_join = df_dates.join(df,['user_id' and df['login_date']==df_dates['all_dates']],'left_anti')
df_join.show()

In [0]:
window_spec = Window.partitionBy('user_id').orderBy('all_dates')

df_lag = df_join\
    .withColumn('lag_date',lag("all_dates").over(window_spec))\
    .withColumn('id',when(col('lag_date').isNull(),1)\
                     .when(datediff("all_dates",'lag_date')>1,1)\
                     .otherwise(0))\
    .withColumn('sum_id',sum('id').over(window_spec))
df_lag.show()

In [0]:
df_result = df_lag.groupBy('user_id','sum_id').agg(
    min('all_dates').alias('missing_from'),
    max('all_dates').alias('missing_to'),
)
df_result.show()